# Forward Curve Modeling Example

This notebook demonstrates how to use the forward curve modeling infrastructure with OKX orderbook data.

We'll cover:
1. PCHIP interpolation with EWMA smoothing
2. Kalman-filtered Nelson-Siegel carry model
3. Evaluation and comparison of both methods


In [2]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
from functools import partial
import numpy as np
import polars as pl

from okx.store import OrderbookStore
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman
from forwards.pchip import reconstruct_forward, PCHIPCurve
from forwards.kalman_ns import reconstruct_ns_forward, NSCarryState
from forwards.evaluation import wmae_pillar_fit, leave_one_expiry_out, evaluate_curve_snapshot


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Initialize Store


In [3]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)


## 2. Build Forward Curves - PCHIP Method

PCHIP (Piecewise Cubic Hermite Interpolating Polynomial) with EWMA smoothing.
Good baseline: simple, stable, interpretable.


In [4]:
# Configure PCHIP recipe with 5-minute bins
pchip_recipe = partial(
    build_forwards_pchip,
    inst_family='BTC-USD',
    binning='5m',
    lambda_ewma=0.8,
    w0_anchor=10.0,
    min_time_to_expiry_hours=2.0,
)

# Build forward curve (cached)
start = datetime(2025, 8, 15)
end = datetime(2025, 8, 16)

lf_pchip = store.get_derived(
    pchip_recipe,
    cache_name='forwards_pchip_5m_test',
    start=start,
    end=end
)

df_pchip = lf_pchip.collect()
print(f"PCHIP forward curve: {df_pchip.shape}")
df_pchip.head(10)


ShapeError: unable to vstack, column names don't match: "inst_type" and "expiry"

### Visualize PCHIP Curve


In [ ]:
import matplotlib.pyplot as plt

# Pick a single snapshot
snapshot_time = df_pchip['timeMs'].unique()[10]
snapshot = df_pchip.filter(pl.col('timeMs') == snapshot_time)

# Separate by source
swap_points = snapshot.filter(pl.col('source') == 'swap')
observed = snapshot.filter(pl.col('source') == 'observed')

fig, ax = plt.subplots(figsize=(12, 6))

# Plot bid/ask curves
ax.plot(snapshot['T'], snapshot['F_bid'], 'b-', label='PCHIP Bid', alpha=0.8)
ax.plot(snapshot['T'], snapshot['F_ask'], 'r-', label='PCHIP Ask', alpha=0.8)

# Mark observed pillars
ax.scatter(observed['T'], observed['F_bid'], c='blue', marker='o', s=50, zorder=5, label='Observed Bid')
ax.scatter(observed['T'], observed['F_ask'], c='red', marker='o', s=50, zorder=5, label='Observed Ask')

# Mark swap anchor
ax.scatter(swap_points['T'], swap_points['F_bid'], c='blue', marker='*', s=200, zorder=5, label='Swap Anchor')
ax.scatter(swap_points['T'], swap_points['F_ask'], c='red', marker='*', s=200, zorder=5)

ax.set_xlabel('Time to Maturity (years)')
ax.set_ylabel('Forward Price (USD)')
ax.set_title(f'PCHIP Forward Curve at {datetime.fromtimestamp(snapshot_time/1000)}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Build Forward Curves - Kalman NS Method

Kalman-filtered Nelson-Siegel carry curve.
More sophisticated: estimates smooth carry curve, handles noise optimally.


In [ ]:
# Configure Kalman NS recipe with 1-minute bins
kalman_recipe = partial(
    build_forwards_kalman,
    inst_family='BTC-USD',
    binning='1m',
    lambda_ns=0.1,
    process_noise_scale=1e-4,
    ar1_coef=0.99,
    min_time_to_expiry_hours=2.0,
)

# Build forward curve (cached)
lf_kalman = store.get_derived(
    kalman_recipe,
    cache_name='forwards_kalman_1m_test',
    start=start,
    end=end
)

df_kalman = lf_kalman.collect()
print(f"Kalman NS states: {df_kalman.shape}")
df_kalman.head(10)


### Reconstruct and Visualize Kalman Curve


In [ ]:
# Pick a single state
state_idx = 60  # 60th minute
state_row = df_kalman[state_idx]

# Create NSCarryState object
state = NSCarryState(
    timeMs=state_row['timeMs'][0],
    beta0=state_row['beta0'][0],
    beta1=state_row['beta1'][0],
    beta2=state_row['beta2'][0],
    lambda_ns=state_row['lambda_ns'][0],
    F_ref_bid=state_row['F_ref_bid'][0],
    F_ref_ask=state_row['F_ref_ask'][0],
)

# Reconstruct curve at various maturities
T_grid = np.linspace(0, 2.0, 100)  # 0 to 2 years
F_bid_grid = reconstruct_ns_forward(state, T_grid, use_bid=True)
F_ask_grid = reconstruct_ns_forward(state, T_grid, use_bid=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot forward curves
ax1.plot(T_grid, F_bid_grid, 'b-', label='Kalman Bid', linewidth=2)
ax1.plot(T_grid, F_ask_grid, 'r-', label='Kalman Ask', linewidth=2)
ax1.scatter([0], [state.F_ref_bid], c='blue', marker='*', s=200, zorder=5, label='Swap Anchor')
ax1.scatter([0], [state.F_ref_ask], c='red', marker='*', s=200, zorder=5)
ax1.set_xlabel('Time to Maturity (years)')
ax1.set_ylabel('Forward Price (USD)')
ax1.set_title(f'Kalman NS Forward Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot NS factors over time
times = df_kalman['timeMs'].to_numpy()
times_dt = [datetime.fromtimestamp(t/1000) for t in times]

ax2.plot(times_dt, df_kalman['beta0'], label='β₀ (Level)', linewidth=2)
ax2.plot(times_dt, df_kalman['beta1'], label='β₁ (Slope)', linewidth=2)
ax2.plot(times_dt, df_kalman['beta2'], label='β₂ (Curvature)', linewidth=2)
ax2.set_xlabel('Time')
ax2.set_ylabel('Factor Value')
ax2.set_title('Nelson-Siegel Factors Evolution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Evaluation - WMAE Pillar Fit

Evaluate how well the curves fit the observed futures prices.


In [ ]:
# Fetch pillar data for evaluation
from okx.recipes.forwards import prepare_pillars

df_pillars = prepare_pillars(
    store,
    inst_family='BTC-USD',
    start=start,
    end=end,
    binning='5m',
    min_time_to_expiry_hours=2.0,
)

print(f"Pillar data: {df_pillars.shape}")
df_pillars.head()


In [ ]:
# Evaluate PCHIP at a specific snapshot
eval_time = df_pillars.filter(pl.col('inst_type') == 'FUTURES')['timeMs'].unique()[5]

# Get observed pillars
pillars_snap = df_pillars.filter(
    (pl.col('timeMs') == eval_time) & (pl.col('inst_type') == 'FUTURES')
)

T_obs = pillars_snap['T'].to_numpy()
F_bid_obs = pillars_snap['bid_1_px'].to_numpy()
F_ask_obs = pillars_snap['ask_1_px'].to_numpy()

# Get PCHIP curve at this time
pchip_snap = df_pchip.filter(pl.col('timeMs') == eval_time)

if not pchip_snap.is_empty():
    # Reconstruct PCHIP curve
    pchip_curve = PCHIPCurve(
        timeMs=int(eval_time),
        T_nodes=pchip_snap['T'].to_numpy(),
        ln_F_bid_nodes=pchip_snap['ln_F_bid'].to_numpy(),
        ln_F_ask_nodes=pchip_snap['ln_F_ask'].to_numpy(),
        symbols=pchip_snap['symbol'].to_list(),
        source=pchip_snap['source'].to_list(),
    )
    
    # Reconstruct at observed maturities
    F_bid_pred, F_ask_pred = reconstruct_forward(pchip_curve, T_obs)
    
    # Evaluate
    rel_spreads = pillars_snap['rel_spread'].to_numpy()
    from forwards.utils import compute_weights
    weights = compute_weights(rel_spreads)
    
    wmae_bid = wmae_pillar_fit(T_obs, F_bid_obs, F_bid_pred, weights)
    wmae_ask = wmae_pillar_fit(T_obs, F_ask_obs, F_ask_pred, weights)
    
    print(f"\nPCHIP Evaluation at {datetime.fromtimestamp(eval_time/1000)}")
    print(f"  Bid WMAE: {wmae_bid.value:.2f} bps")
    print(f"  Ask WMAE: {wmae_ask.value:.2f} bps")
    print(f"  Bid MAE: {wmae_bid.details['mae_bps']:.2f} bps")
    print(f"  Ask MAE: {wmae_ask.details['mae_bps']:.2f} bps")
else:
    print("No PCHIP curve data at this timestamp")


## 5. Compare PCHIP vs Kalman

Visual comparison of both methods.


In [ ]:
# Track a specific forward (e.g., 3-month) over time
T_track = 0.25  # 3 months

# PCHIP tracking
pchip_tracking = []
for time_ms in df_pchip['timeMs'].unique():
    snap = df_pchip.filter(pl.col('timeMs') == time_ms)
    if not snap.is_empty():
        curve = PCHIPCurve(
            timeMs=int(time_ms),
            T_nodes=snap['T'].to_numpy(),
            ln_F_bid_nodes=snap['ln_F_bid'].to_numpy(),
            ln_F_ask_nodes=snap['ln_F_ask'].to_numpy(),
            symbols=snap['symbol'].to_list(),
            source=snap['source'].to_list(),
        )
        F_bid, F_ask = reconstruct_forward(curve, T_track)
        pchip_tracking.append({
            'timeMs': time_ms,
            'F_mid': (F_bid + F_ask) / 2
        })

df_pchip_track = pl.DataFrame(pchip_tracking)

# Kalman tracking
kalman_tracking = []
for row in df_kalman.iter_rows(named=True):
    state = NSCarryState(
        timeMs=row['timeMs'],
        beta0=row['beta0'],
        beta1=row['beta1'],
        beta2=row['beta2'],
        lambda_ns=row['lambda_ns'],
        F_ref_bid=row['F_ref_bid'],
        F_ref_ask=row['F_ref_ask'],
    )
    F_bid = reconstruct_ns_forward(state, T_track, use_bid=True)
    F_ask = reconstruct_ns_forward(state, T_track, use_bid=False)
    kalman_tracking.append({
        'timeMs': row['timeMs'],
        'F_mid': (F_bid + F_ask) / 2
    })

df_kalman_track = pl.DataFrame(kalman_tracking)

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

times_pchip = [datetime.fromtimestamp(t/1000) for t in df_pchip_track['timeMs']]
times_kalman = [datetime.fromtimestamp(t/1000) for t in df_kalman_track['timeMs']]

ax.plot(times_pchip, df_pchip_track['F_mid'], label='PCHIP', linewidth=2, alpha=0.8)
ax.plot(times_kalman, df_kalman_track['F_mid'], label='Kalman NS', linewidth=2, alpha=0.8)

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Forward Price (USD)', fontsize=12)
ax.set_title(f'{T_track*12:.1f}-Month Forward Price Evolution', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Compute smoothness
pchip_changes = np.abs(np.diff(np.log(df_pchip_track['F_mid'].to_numpy())))
kalman_changes = np.abs(np.diff(np.log(df_kalman_track['F_mid'].to_numpy())))

print(f"\nSmoothness Analysis (3-month forward):")
print(f"  PCHIP - Mean absolute log change: {pchip_changes.mean():.6f}")
print(f"  PCHIP - Std of log changes: {pchip_changes.std():.6f}")
print(f"  Kalman - Mean absolute log change: {kalman_changes.mean():.6f}")
print(f"  Kalman - Std of log changes: {kalman_changes.std():.6f}")
